# Ⅰ第1回 GW3　コードを組み立ててモデルを作る（教科書 2.3 節 コードテンプレート①〜⑫）

**今日の問い**：ディープラーニングのプログラムはどんな順番で何をしているか

このノートブックには，教科書 2.3 節のコードテンプレート①〜⑫（FashionMNIST を MLP で分類する完成コード）が
**順番をばらばらにして** 入っている．番号も伏せてある．

**やること**
1. GW2 で並べた「ディープラーニングプログラミングの流れ」（データ準備 → 学習設定 → 学習・評価）を手がかりに，
   各セルの `# コードテンプレート ?` の `?` に番号①〜⑫を書き込む（コードを読んで判断する）
2. セルを正しい順に並べ替える（VS Code：セル左のつまみをドラッグ，または セルを選んで Alt+↑ / Alt+↓）
3. 上から順に実行する．順番が違うと `NameError` などで止まる．**どのセルがどのセルに依存しているか** をエラーから読み取る
4. 最後まで通ったら，学習曲線（⑪）とテストデータの正解率（⑫）を確認し，最終セルで CSV に記録する

**注意**：LLM に「正しい順に並べて」と頼めば答えは出る．しかしこの後の演習で使うのは，
「このコードは何をしているか」を自分で説明できることである．先に読み，迷ったところだけ LLM に聞く．


## (0) 班と役割の設定

In [ ]:
# ===== (0) 班と役割の設定 =====
GROUP_ID = 1                  # ← 自分の班番号 (1〜27) に書き換える
MEMBER_ROLE = "implementer"   # implementer / verifier / recorder / presenter（5 人班は collector も）のいずれか．3 人班で presenter を兼ねる verifier は "verifier"

# 授業用フォルダ（AI_TD）のルートを import パスに追加する（ノートブックをどこから開いても動く）
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".dlcourse_root").exists())
sys.path.insert(0, str(ROOT))
print("作業フォルダ:", ROOT)

## 並び替えるセル（12 個）

各セルの `?` に番号を書き，正しい順に並べ替えてから実行する．

In [ ]:
# コードテンプレート ?
criterion = nn.CrossEntropyLoss()

In [ ]:
# コードテンプレート ?
data_transform = transforms.Compose([
    transforms.ToTensor(),
])

In [ ]:
# コードテンプレート ?
imgs, _ = next(iter(train_loader))
c, h, w = imgs[0].shape
print("ミニバッチサイズ:", len(imgs))
print("チャネル数:", c)
print("画像の高さ:", h)
print("画像の幅:", w)
print(imgs.shape)

img = torchvision.utils.make_grid(imgs)
img = transforms.functional.to_pil_image(img)
display(img)

In [ ]:
# コードテンプレート ?
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"使用デバイス: {device}")

In [ ]:
# コードテンプレート ?
EPOCHS = 3          # 授業用に 3 エポック（教科書は 10）
train_loss_list = []
val_loss_list = []
train_acc_list = []
val_acc_list = []

import time
_t0 = time.perf_counter()
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]', leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()
    
    train_loss /= train_total
    train_accuracy = 100.0 * train_correct / train_total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_accuracy)
    
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Val]', leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    
    val_loss /= val_total
    val_accuracy = 100.0 * val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_accuracy)

    print(f'Epoch {epoch+1}/{EPOCHS},'
          f'Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.2f}%,'
          f'Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.2f}%  ')

train_sec = time.perf_counter() - _t0
print(f'学習時間: {train_sec:.1f} 秒')

In [ ]:
# コードテンプレート ?
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# コードテンプレート ?
INPUT_SIZE = c * h * w
HIDDEN_SIZE1 = 512
HIDDEN_SIZE2 = 256
OUTPUT_SIZE = 10

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(INPUT_SIZE, HIDDEN_SIZE1),
    nn.ReLU(),
    nn.Linear(HIDDEN_SIZE1, HIDDEN_SIZE2),
    nn.ReLU(),
    nn.Linear(HIDDEN_SIZE2, OUTPUT_SIZE)
)
model.to(device)

model

In [ ]:
# コードテンプレート ?
model
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
print(f'Accuracy: {correct / total * 100:.2f}% ')

In [ ]:
# コードテンプレート ?
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed(0)
if torch.backends.mps.is_available():
    torch.mps.manual_seed(0)

train_val_dataset = datasets.FashionMNIST(
    root=str(ROOT / 'assets' / 'data'), 
    train=True, 
    download=True, 
    transform=data_transform
)
test_dataset = datasets.FashionMNIST(
    root=str(ROOT / 'assets' / 'data'),
    train=False,
    download=True,
    transform=data_transform
)

train_size = int(0.8 * len(train_val_dataset))
val_size = len(train_val_dataset) - train_size
train_dataset, val_dataset = random_split(
    train_val_dataset, 
    [train_size, val_size]
)
print(f"訓練データ数: {len(train_dataset)} 枚)")
print(f"検証データ数: {len(val_dataset)} 枚)")
print(f"テストデータ数: {len(test_dataset)} 枚)")

In [ ]:
# コードテンプレート ?
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# コードテンプレート ?
import seaborn as sns
import matplotlib.pyplot as plt
epochs = range(1, len(train_loss_list) + 1)
plt.figure(figsize=(10, 5))

sns.lineplot(x=epochs, y=train_loss_list, label='Train Loss')
sns.lineplot(x=epochs, y=val_loss_list, label='Val Loss')
plt.title('Loss over epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
sns.lineplot(x=epochs, y=train_acc_list, label='Train Accuracy')
sns.lineplot(x=epochs, y=val_acc_list, label='Val Accuracy')
plt.title('Accuracy over epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# コードテンプレート ?
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import random
import numpy as np

## 記録と討議

**CSV に記録**：下のセルで，テストデータの正解率と学習時間を共通形式の CSV に残す（演習1〜3 と同じ形式）．

**グループディスカッション（5 分）**
1. 学習が始まるのは何番目のセルか．それより前の 9 個のセルは何をしているか，一文で言う
2. データが画像からテキストに変わったとき，差し替えるのはどのセルか（→ 演習2 で確認する）
3. `NameError` で止まったセルがあれば，「どのセルが先に必要だったか」を記録する
4. 正解率は班の 4 台で同じ値になったか．違うなら何が原因か（→ 第4回で扱う）


In [ ]:
from common.logger import ResultLogger
logger = ResultLogger(GROUP_ID, MEMBER_ROLE, course="c1", day="d1", exercise="ex0", device=device)
logger.log_many({"test_accuracy": correct / total, "epochs": EPOCHS, "train_sec": train_sec},
                condition="FashionMNIST_MLP", seed=0)
print("書き込み先:", logger.path)
print(f"テスト正解率 {correct / total * 100:.2f}%  学習時間 {train_sec:.1f} 秒  エポック {EPOCHS}")